# pocketbench example: benchmark and conformance

This notebook drives pocketbench through its programmatic facade, `pocketbench.api.PocketBench`:
the same class the `pocket-bench` CLI uses, so notebook results can never diverge from CLI results.

It shows two operations end to end:

1. **Benchmark** (`bench`): the uniform in-process compress/decompress timing loop, with all
   metrics derived in core (time per packet, packets/sec, MB/s).
2. **Conformance** (`conformance`): grading an implementation against the UAB/CNES
   cross-validation suite (size + SHA-256 per vector versus `file_list.csv`).

## Prerequisites

- Run from the project's uv environment: `uv sync --extra notebook`, then start Jupyter with
  `uv run --with jupyter jupyter lab` (or point your kernel at `.venv`). The `notebook` extra
  provides pandas for `.to_dataframe()` and jinja2 for the styled result tables.
- The codec sources must be present under `implementations/` (see `implementations/README.md`);
  the adapters are built from them on demand.
- For the conformance section only: `[settings] conformance_data_dir` in `config/config.toml`
  must point at the extracted UAB/CNES suite. The suite is local-only and cannot be committed.

The facade never prints on its own and writes nothing to disk unless asked: progress arrives via
callbacks, expected failures come back as data on the result containers, and persistence is the
explicit `write_report()` call.

In [1]:
from pathlib import Path

from pocketbench.api import PocketBench

# The notebook lives in notebooks/, the default config one level up.
CONFIG = Path("..") / "config" / "config.toml"


def on_message(msg: str) -> None:
    print(msg)


def on_progress(phase: str, done: int, total: int) -> None:
    # Fired every 1000 conformance vectors, so a full run prints a handful of lines.
    print(f"{phase}: {done}/{total}")


pb = PocketBench(CONFIG, on_message=on_message, on_progress=on_progress)

print("implementations:", [a.manifest.name for a in pb.config.adapters])
print("datasets:       ", list(pb.config.datasets))

implementations: ['pocketrust', 'reference-c', 'reference-cpp', 'reference-rust']
datasets:        ['reference-edge-cases', 'reference-hiro', 'reference-housekeeping', 'reference-simple', 'reference-venus-express', 'synthetic-f1']


## Build the adapters

Each implementation is fronted by a committed adapter under `adapters/<name>/`. `build()` compiles
the selected adapters (gate 1: the codec source must be present). This step is optional here:
`bench()` and `conformance()` build by default, but running it separately makes build errors easy
to see, and the returned `BuildRun` carries per-impl stderr.

This example sticks to the two ESA reference implementations. Swap in any other discovered adapter
name (printed above) to compare more.

In [2]:
IMPLS = ['pocketrust', 'reference-c', 'reference-cpp', 'reference-rust']

build_run = pb.build(IMPLS)
print("all builds ok:", build_run.ok)
build_run.to_dataframe()

building pocketrust
building reference-c
building reference-cpp
building reference-rust
all builds ok: True


,impl,ok,stderr
0,pocketrust,True,[1m[92m Finished[0m ]8;;https://doc.rus...
1,reference-c,True,
2,reference-cpp,True,
3,reference-rust,True,[1m[92m Finished[0m ]8;;https://doc.rus...


## Run the benchmark

`bench()` stages each selected dataset and calls the adapter's in-process timed loop
(`warmup` untimed iterations, then `iterations` timed ones); core derives every metric from the
raw per-iteration nanoseconds, so all implementations share one code path.

Notes:

- The reference datasets (`reference-simple`, `reference-housekeeping`, ...) read their
  compression parameters from each vector's `expected-output/*-metadata.json`, so they cannot
  drift from the reference output.
- For the C implementation, `bench` builds the codec with `CCSDS124_MAX_PACKET_LENGTH` sized to
  the largest configured dataset instead of the 65535-bit default: the stock build memsets a
  ~96 KB buffer per 90-byte packet and measures roughly 5x slower than it should.
- Absolute numbers are machine-specific; comparisons between implementations on the same run are
  the meaningful output.

In [3]:
bench_run = pb.bench(
    IMPLS,
    datasets=['reference-edge-cases', 'reference-hiro', 'reference-housekeeping', 'reference-simple', 'reference-venus-express', 'synthetic-f1'],
    warmup=10,
    iterations=100,
    build=False,
)

bench_df = bench_run.to_dataframe()
bench_df

running pocketrust bench (100 iter, 10 warmup)
running reference-c bench (100 iter, 10 warmup)
running reference-cpp bench (100 iter, 10 warmup)
reference-cpp/synthetic-f1: reference-cpp: bench compress failed
unsupported packet_bits 1 (compile-time template; see SUPPORTED_SIZES)
running reference-rust bench (100 iter, 10 warmup)
reference-rust/synthetic-f1: reference-rust: bench compress failed
bench codec error: invalid packet size: 1 (must be > 0 and divisible by 8)


,impl,vector,operation,num_packets,size_bytes,time_ms,us_per_pkt,packets_per_sec,mb_per_sec
0,pocketrust,reference-edge-cases,compress,500,45000,0.229696,0.459392,2.176790e+06,186.835401
1,pocketrust,reference-edge-cases,decompress,500,45000,0.211794,0.423588,2.360785e+06,202.627762
2,pocketrust,reference-hiro,compress,100,9000,0.048042,0.480415,2.081534e+06,178.659468
3,pocketrust,reference-hiro,decompress,100,9000,0.046104,0.461040,2.169009e+06,186.167553
4,pocketrust,reference-housekeeping,compress,10000,900000,4.882534,0.488253,2.048117e+06,175.791259
5,pocketrust,reference-housekeeping,decompress,10000,900000,5.835484,0.583548,1.713654e+06,147.084083
6,pocketrust,reference-simple,compress,100,9000,0.054426,0.544255,1.837374e+06,157.703078
7,pocketrust,reference-simple,decompress,100,9000,0.031466,0.314655,3.178084e+06,272.777132
8,pocketrust,reference-venus-express,compress,151200,13608000,90.621840,0.599351,1.668472e+06,143.206098
9,pocketrust,reference-venus-express,decompress,151200,13608000,101.864058,0.673704,1.484331e+06,127.401169


The dataframe has one row per (implementation, vector, operation), with the BENCHMARK.md columns
(`us_per_pkt`, `packets_per_sec`, `mb_per_sec`). A pivot makes the cross-implementation comparison
readable, and the fastest implementation per (vector, operation) is highlighted in yellow, matching
the CLI's benchmark tables (higher is better for `mb_per_sec`, lower for `us_per_pkt`):

In [4]:
import pandas as pd

# Highlight the fastest implementation per (vector, operation), like the CLI's yellow row.
# Within a group all impls process the same packets, so max mb_per_sec picks the same impl
# the CLI marks (it ranks by packets_per_sec); it is highlighted under both metrics.
FASTEST = "background-color: #ffd54f; color: black; font-weight: bold;"

pivot = bench_df.pivot_table(
    index=["vector", "operation"],
    columns="impl",
    values=["mb_per_sec", "us_per_pkt"],
)


def highlight_fastest(df: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for row in df.index:
        winner = df.loc[row, "mb_per_sec"].idxmax()
        for metric in ("mb_per_sec", "us_per_pkt"):
            styles.loc[row, (metric, winner)] = FASTEST
    return styles


pivot.style.format("{:.3f}").apply(highlight_fastest, axis=None)

Persisting results is opt-in. `write_report()` writes `benchmark.md` (a BENCHMARK.md replica,
including a Build Settings section for reproducibility) and `benchmark.json` to
`settings.results_dir`:

In [ ]:
for path in pb.write_report(bench_run):
    print("wrote", path)

## Run the conformance test

`conformance()` runs an implementation against the UAB/CNES suite: every vector goes through the
adapter's `conformance-compress` / `conformance-decompress` verb and its output is compared by size
and SHA-256 to `crossvalidation/file_list.csv`. The full suite is 7,935 encoder + 16,965 decoder
vectors, so this demo uses `limit=` for a quick smoke test; drop it for a real run.

It **reports, it does not grade**:

- `mode` is `"encoder"`, `"decoder"`, or `"both"`.
- Only implementations whose adapter self-reports conformance support run; others are skipped
  with a message (e.g. `reference-rust`, whose codec crate keeps the per-packet API private).
- You get `total` / `passed` / `failed` per implementation plus the failing vector names. There is
  no pass/fail verdict and no per-impl strictness: comparing the counts against expectations is
  your job.
- The adapters are built here at the codec-default full 65535-bit packet size (`build=True`),
  because UAB vectors have `large_f` up to 65535, unlike the smaller bench build above.

In [5]:
conf_run = pb.conformance(
    ["pocketrust"],
    mode="both",
    # limit=200,  # smoke test; remove for the full suite
)

conf_run.to_dataframe()

running pocketrust conformance (both, ccsds124_full_crossvalidation)
encoder: 1000/7935
encoder: 2000/7935
encoder: 3000/7935
encoder: 4000/7935
encoder: 5000/7935
encoder: 6000/7935
encoder: 7000/7935
decoder: 1000/16965
decoder: 2000/16965
decoder: 3000/16965
decoder: 4000/16965
decoder: 5000/16965
decoder: 6000/16965
decoder: 7000/16965
decoder: 8000/16965
decoder: 9000/16965
decoder: 10000/16965
decoder: 11000/16965
decoder: 12000/16965
decoder: 13000/16965
decoder: 14000/16965
decoder: 15000/16965
decoder: 16000/16965


,impl,mode,total,passed,failed
0,pocketrust,both,24900,24891,9


Each `ConformanceResult` carries the individual failing vector names in `.failures`,
so a failure can be taken straight to `explain()` below:

In [6]:
for r in conf_run.results:
    print(f"{r.impl} [{r.mode}] {r.passed}/{r.total} passed, {r.failed} failed")
    for name in r.failures[:10]:
        print("   ", name)
    if len(r.failures) > 10:
        print(f"    ... {len(r.failures) - 10} more")

pocketrust [both] 24891/24900 passed, 9 failed
    decoder_sequence_09048.raw+large_f
    decoder_sequence_09336.raw+large_f
    decoder_sequence_09337.raw+large_f
    decoder_sequence_10346.raw+large_f
    decoder_sequence_10395.raw+large_f
    decoder_sequence_12725.raw+large_f
    decoder_sequence_14972.raw+large_f
    decoder_sequence_15361.raw+large_f
    decoder_sequence_15681.raw+large_f


# Testing

## Explain a failing vector

`conformance()` says *which* vectors failed; `explain()` says *what* diverged, by diffing the
adapter's output against the suite's expected output bytes.

Decoder vectors get a **frame**-level diff (a frame is one status byte plus, when decoded,
`large_f` bits padded to the byte). Encoder vectors get a byte-level diff, because encoder output is
an unframed concatenation of compressed packets.

`show_frame(i, mode="hex" | "bin")` prints one frame against ground truth with the differing
bytes/bits marked; `first_divergence` is the frame to start with.

In [ ]:
failing = [(r.impl, name) for r in conf_run.results for name in r.failures]

if not failing:
    print("no failures to explain")
else:
    impl, vector = failing[0]
    exp_run = pb.explain(vector, [impl], build=False)
    print(exp_run.summary())

    r = exp_run[0]
    if r.decoder is not None and r.decoder.first_divergence is not None:
        print()
        print(r.show_frame(r.decoder.first_divergence, mode="hex"))

`ExplainRun.to_dataframe()` gives one row per diverging frame, so several vectors can be
compared at once: